# Week 12: `pandas` data manipulation and introduction to visualization

This week builds on last week's introduction to NumPy and pandas. We will read real datasets, inspect and clean data, select rows and columns, create new variables, summarize observations with `groupby()`, and make plots using `matplotlib` and `seaborn`.

**Files:**  `states_covid.csv`, `muricata_pops_lat_long.csv`, and `Bloom_etal_2018_Reduced_Dataset.csv` (the latter two are used in optional examples). Place them alongside this notebook. The COVID Tracking Project data are historical, ending in early 2021, and reporting conventions differ among states. 

**Learning goals:** use `.loc` and `.iloc`; filter with Boolean conditions; manage missing values and dates; summarize with `groupby()`; distinguish cumulative from daily counts; and produce readable figures.

## I. Import libraries and read a dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Use the filename provided for this week.
covid_df = pd.read_csv("states_covid.csv")
covid_df.head()

A **DataFrame** is a two-dimensional table with named columns and a row index. `read_csv()` loads data; `.head()` shows the first five rows. `shape` gives (rows, columns), while `.info()` helps identify column types and missing values.

In [ ]:
print("Rows and columns:", covid_df.shape)
print("Columns:", covid_df.columns.tolist())
covid_df.info()

### Select rows and columns

Use one pair of brackets for a Series (`df["death"]`) and double brackets for a DataFrame (`df[["state", "death"]]`). `.loc` selects by **labels**; `.iloc` selects by **integer position**. Setting `state` as an index is convenient, but the index can have repeated labels (one row per state and date).

In [ ]:
covid_df[["date", "state", "death", "positive"]].head()


In [ ]:
indexed_df = covid_df.set_index("state")
indexed_df.loc["NV", ["date", "death"]].head()

In [ ]:
covid_df.iloc[:5, :4]  # first five rows, first four columns

## II. Clean data and make new variables

Dates initially look like integers (e.g., `20210223`). Convert them to datetime objects before sorting or plotting. `.copy()` makes an independent subset, avoiding ambiguous assignments to views.

In [ ]:
working_df = covid_df[["date", "state", "death", "deathIncrease", "positive", "positiveIncrease", "totalTestResults"]].copy()
# Convert dates to pandas datetime format
working_df["date"] = pd.to_datetime(working_df["date"])
# Rename column
working_df = working_df.rename(columns={"totalTestResults": "total_tests"})
# Extract day of year
working_df["dayofyear"] = working_df["date"].dt.dayofyear
working_df.head()

### Missing data and derived columns

Missing values are generally represented by `NaN`. Check them with `.isna()` and count them with `.sum()`. A missing value is **not automatically zero**. The cumulative `positive` and `death` columns are not interchangeable with daily `positiveIncrease` and `deathIncrease`. State reporting definitions also differ; avoid interpreting `positive / total_tests` as a consistent diagnostic positivity rate across states.

In [ ]:
working_df.isna().sum()

In [ ]:
working_df["cases_per_1000_tests"] = 1000 * working_df["positive"] / working_df["total_tests"].replace(0, np.nan)
working_df[["state", "date", "cases_per_1000_tests"]].head()
# This is a descriptive ratio of cumulative reported measures, NOT a standardized test positivity rate.

## III. Filter, sort, and export

In [ ]:
nv_df = working_df[working_df["state"] == "NV"].copy()
nv_df = nv_df.sort_values("date")
nv_df.head()

Boolean filters use `&` for AND and `|` for OR. Put each condition in parentheses. The `.query()` method provides an alternative. Sorting by date is essential before plotting a time series.

In [ ]:
nv_high_df = working_df[(working_df["state"] == "NV") & (working_df["deathIncrease"] > 20)]
nv_high_df[["date", "state", "deathIncrease"]].head()

In [ ]:
working_df.query("state == 'NV' and deathIncrease > 20").head()

In [ ]:
top_days_df = nv_df.sort_values("deathIncrease", ascending=False).head(5)
top_days_df[["date", "deathIncrease"]]

In [ ]:
# Uncomment to write the selected data to a new CSV file.
# nv_df.to_csv("nevada_covid_subset.csv", index=False)

## IV. Summarize with `groupby()` and `.agg()`

`groupby()` splits a table into groups, applies calculations, and combines the results. Use named aggregations to keep output column names simple. **Important:** summing cumulative counts across dates is usually meaningless. For state-level cumulative totals, select the latest available record per state; for daily reported counts, sums or means over a specified period can be useful.

In [ ]:
state_summary_df = (
    working_df.sort_values("date")
    .groupby("state", as_index=False)
    .agg(latest_date=("date", "max"),
         mean_daily_deaths=("deathIncrease", "mean"),
         max_daily_deaths=("deathIncrease", "max"),
         observed_days=("date", "count"))
)
state_summary_df.sort_values("mean_daily_deaths", ascending=False).head()

In [ ]:
latest_state_df = (working_df.sort_values("date")
                   .drop_duplicates(subset="state", keep="last"))
latest_state_df[["state", "date", "death", "positive"]].head()

### Group by weekday (Nevada)

The `deathIncrease` field represents daily reported changes, while `death` is cumulative. Grouping cumulative `death` by weekday and summing would count the same deaths repeatedly. Weekday patterns can reflect reporting delays rather than biological differences.

In [ ]:
nv_df["weekday"] = nv_df["date"].dt.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_df = (nv_df.groupby("weekday")["deathIncrease"]
              .agg(["mean", "count"])
              .reindex(weekday_order))
weekday_df

### Optional: group biological data

If you have `Bloom_etal_2018_Reduced_Dataset.csv`, the same approach summarizes ecological variables by `Reg`. Named aggregations avoid the MultiIndex column names created by aggregating several statistics for several columns.

In [ ]:
# Optional: uncomment if the Bloom dataset is in this folder.
# bloom_df = pd.read_csv("Bloom_etal_2018_Reduced_Dataset.csv")
# bloom_df.groupby("Reg").agg(mean_body_size=("logbodysize", "mean"),
#                              n=("logbodysize", "count"))

## V. Introduction to plotting with `matplotlib`

A figure (`fig`) contains one or more axes (`ax`). We will use `fig, ax = plt.subplots()` and call plotting methods on `ax`. Add axis labels and a title, and use `plt.show()` to display the figure. `matplotlib` offers fine control; `seaborn` provides convenient statistical graphics built on matplotlib.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(nv_df["date"], nv_df["death"], linewidth=2)
ax.set(title="Nevada: cumulative reported deaths", xlabel="Date", ylabel="Cumulative deaths")
fig.autofmt_xdate()
plt.show()

A daily series is often noisy. A 7-day rolling mean can make broad trends easier to see. `rolling(7).mean()` averages seven successive observations (after sorting by date).

In [ ]:
nv_df["deaths_7day_mean"] = nv_df["deathIncrease"].rolling(7, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(nv_df["date"], nv_df["deathIncrease"], alpha=0.35, label="Daily reported increase")
ax.plot(nv_df["date"], nv_df["deaths_7day_mean"], linewidth=2, label="7-day mean")
ax.set(title="Nevada: daily reported deaths", xlabel="Date", ylabel="Reported deaths per day")
ax.legend()
fig.autofmt_ 
xdate()
plt.show()

## VI. Introduction to `seaborn`

Seaborn works naturally with **long-format** DataFrames: each row is an observation and columns specify variables. Use `data=`, `x=`, `y=`, and sometimes `hue=`. For comparing states, choose a few rather than plotting all 50+ series together.

In [ ]:
selected_df = working_df[working_df["state"].isin(["NV", "CA", "OR", "AZ"])].copy()
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=selected_df, x="date", y="deathIncrease", hue="state", ax=ax, estimator=None)
ax.set(title="Daily reported deaths in four states", xlabel="Date", ylabel="Daily reported deaths")
fig.autofmt_xdate()
plt.show()  


A boxplot compares distributions. Here each observation is a daily reported increase, not a cumulative total. A negative value can occur when historical records are revised.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=selected_df, x="state", y="deathIncrease", ax=ax)
ax.set(title="Distribution of daily reported deaths",
       xlabel="State",
       ylabel="Daily reported deaths")

plt.show()

A bar chart is useful for a small grouped summary. Here we use **means of daily increases**, rather than summing cumulative totals.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=weekday_df.reset_index(), x="weekday", y="mean", ax=ax)
ax.set(title="Nevada: mean reported daily deaths by weekday", xlabel="Weekday", ylabel="Mean daily reported deaths")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Saving figures

Save **before** `plt.show()` (or call `fig.savefig()` while the figure still exists). PDF and SVG are useful for scalable figures; PNG is useful for slides.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(nv_df["date"], nv_df["deaths_7day_mean"])
ax.set(title="Nevada: seven-day mean of reported deaths", xlabel="Date", ylabel="Reported deaths per day")
fig.autofmt_xdate()
# fig.savefig("nv_daily_deaths.pdf", bbox_inches="tight")
plt.show()

### Seaborn: Violin plots

Violin plots are similar to boxplots but show more information about the shape of a distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(
    data=selected_df,
    x="state",
    y="deathIncrease",
    palette="viridis",
    inner="quartile",
    cut=0,
    ax=ax
)

ax.set(
    title="Distribution of daily reported COVID deaths",
    xlabel="State",
    ylabel="Daily reported deaths"
)

sns.despine()
plt.tight_layout()
plt.show()

### Seaborn: Small multiples with FacetGrid

This is one of my favorite approaches for comparing patterns across groups.

Rather than putting every state on one figure, we create separate panels.

In [ ]:
g = sns.relplot(
    data=selected_df,
    x="date",
    y="deathIncrease",
    col="state",
    col_wrap=3,
    kind="line",
    height=3,
    aspect=1.3,
    color="steelblue",
    facet_kws={"sharey": False}
)
g.set_axis_labels("Date", "Daily deaths")
g.set_titles("{col_name}")
g.fig.suptitle(
    "Daily COVID deaths across states",
    y=1.03,
    fontsize=16
)
plt.show()

### Seaborn: Heatmap of correlations

A very useful general-purpose visualization for multivariate biological datasets.

In [ ]:
# Select numerical variables
variables = [
    "death",
    "deathIncrease",
    "positive",
    "positiveIncrease",
    "total_tests"
]
# Calculate correlations
correlations = working_df[variables].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    correlations,
    annot=True,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    fmt=".2f",
    ax=ax
)
ax.set_title("Correlations among COVID variables")
plt.tight_layout()
plt.show()

In [ ]:
# Calculate average daily deaths by state and month
monthly_df = selected_df.copy()
monthly_df["month"] = monthly_df["date"].dt.to_period("M").astype(str)
monthly_summary = monthly_df.groupby(
    ["state", "month"]
)["deathIncrease"].mean().reset_index()
# Reshape to a matrix
heatmap_df = monthly_summary.pivot(
    index="state",
    columns="month",
    values="deathIncrease"
)
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    heatmap_df,
    cmap="magma",
    linewidths=0.3,
    cbar_kws={"label": "Mean daily deaths"},
    ax=ax
)
ax.set(
    title="Temporal variation in COVID deaths",
    xlabel="Month",
    ylabel="State"
)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Seaborn: Pairplot

A particularly powerful exploratory visualization.

In [ ]:
# Select a few variables and a manageable subset of observations
pair_df = selected_df[
    ["state", "deathIncrease", "positiveIncrease", "total_tests"]
].dropna()

pair_df = pair_df.sample(
    n=min(500, len(pair_df)),
    random_state=42
)
sns.pairplot(
    pair_df,
    vars=["deathIncrease", "positiveIncrease", "total_tests"],
    hue="state",
    corner=True,
    plot_kws={"alpha": 0.5, "s": 20}
)
plt.show()

### Interactive visualization with Plotly

Plotly introduces interactive graphics that students can explore in Jupyter notebooks or export as HTML.

First, import the library:

In [ ]:
import plotly.express as px

### plotly: interactive scatter plot

In [ ]:
fig = px.scatter(
    selected_df,
    x="positiveIncrease",
    y="deathIncrease",
    color="state",
    hover_data=["date"],
    title="Daily COVID cases and deaths",
    labels={
        "positiveIncrease": "Daily positive cases",
        "deathIncrease": "Daily deaths"
    },
    opacity=0.5
)

fig.update_layout(
    template="plotly_white",
    width=900,
    height=550
)

fig.show()

### plotly: interactive time series

In [ ]:
fig = px.line(
    selected_df.sort_values("date"),
    x="date",
    y="deathIncrease",
    color="state",
    title="Daily COVID deaths by state",
    labels={
        "date": "Date",
        "deathIncrease": "Daily deaths"
    }
)
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    width=1000,
    height=550
)
fig.show()

### Plotly: Animated geographic map

This is one of the more visually engaging examples.

We can animate changes in COVID cases across the United States.

In [ ]:
map_df = working_df.copy()
# Select only dates with observations for at least 50 states
state_counts = map_df.groupby("date")["state"].transform("nunique")
map_df = map_df[state_counts >= 50].copy()
# Select Mondays to reduce animation size
map_df = map_df[map_df["date"].dt.dayofweek == 0].copy()
# Convert dates to strings for animation
map_df["date_string"] = map_df["date"].dt.strftime("%Y-%m-%d")
# Sort chronologically
map_df = map_df.sort_values("date")
# Create animated map
fig = px.choropleth(
    map_df,
    locations="state",
    locationmode="USA-states",
    color="positiveIncrease",
    animation_frame="date_string",
    scope="usa",
    color_continuous_scale="Viridis",
    range_color=(0, 20000),
    title="Daily reported COVID cases across the United States"
)
fig.update_layout(
    width=1000,
    height=650,
    geo=dict(bgcolor="rgba(0,0,0,0)")
)
fig.show()

 ### Plotly: Static choropleth map

For a simpler map, we can summarize the dataset by state.

In [ ]:
state_summary = working_df.groupby(
    "state"
)["positive"].max().reset_index()
fig = px.choropleth(
    state_summary,
    locations="state",
    locationmode="USA-states",
    color="positive",
    scope="usa",
    color_continuous_scale="Plasma",
    title="Maximum reported cumulative COVID cases by state",
    labels={"positive": "Cumulative cases"}
)
fig.update_layout(
    width=950,
    height=600
)
fig.show()

## VII. Try it yourself (optional practice)

1. Select data for two states of your choice and report their date ranges.
2. Find the five dates with the largest `positiveIncrease` for Nevada.
3. Use `groupby()` to calculate the mean daily reported cases (`positiveIncrease`) by state.
4. Plot the daily cases and a seven-day rolling mean for one state.
5. Make a labeled seaborn boxplot comparing daily cases across three states.

**Questions to consider:** What happens if you use cumulative `positive` instead of daily `positiveIncrease`? What does `NaN` mean in this dataset? How might reporting delays affect weekday patterns?

## Appendix: quick reference

| Task | Example |
|---|---|
| Load CSV | `pd.read_csv("file.csv")` |
| Inspect | `df.head()`, `df.shape`, `df.info()` |
| Select columns | `df[["state", "death"]]` |
| Select rows by label | `df.loc["NV"]` (after indexing by state) |
| Select by position | `df.iloc[:5, :3]` |
| Filter | `df[df["state"] == "NV"]` |
| Sort | `df.sort_values("date")` |
| Missing data | `df.isna().sum()` |
| Dates | `pd.to_datetime(...)`, `df["date"].dt.day_name()` |
| Group | `df.groupby("state")["deathIncrease"].mean()` |
| Save CSV | `df.to_csv("output.csv", index=False)` |
| Make figure | `fig, ax = plt.subplots()` |
| Plot | `ax.plot(x, y)`, `sns.boxplot(data=df, x="state", y="deathIncrease")` |
| Save figure | `fig.savefig("figure.pdf", bbox_inches="tight")` |

**Optional further exploration:** `sns.scatterplot()`, `sns.histplot()`, `sns.relplot()` and `plotly.express` for interactive graphics. The focus this week is on understanding DataFrame operations and producing a few clear, interpretable figures.